# Evaluating the Impact of Image Augmentation Strategies on CNN Architectures in Skin Lesion Classification

**DS 6050, SP 2026 - ML III: Deep Learning Project** *For Professor Heman Shakeri, PhD, University of Virginia*

---

### Team Members
* **Robert Ashby** | *University of Virginia, School of Data Science* | Fernandina Beach, Florida | gsr3qz@virginia.edu
* **Xavier Colbert** | *University of Virginia, School of Data Science* | Alexandria, Virginia | kxp3jj@virginia.edu
* **Jacob Kuchta** | *University of Virginia, School of Data Science* | Arlington, Virginia | mjk3ku@virginia.edu
* **Alysa Pugmire** | *University of Virginia, School of Data Science* | Richmond, Virginia | amp3xs@virginia.edu

## Notebook purpose

This notebook evaluates phase-specific augmentation strategies for skin lesion classification using three CNN backbones:

- ResNet-50
- EfficientNet-B0
- DenseNet-121

The objective is to measure the isolated effect of each augmentation phase relative to the Phase 1 baseline established in `02_baseline_modeling.ipynb`.

## Experimental structure

The notebook is organized around three augmentation experiments:

- `phase2_only` — geometric transforms
- `phase3_only` — color-based transforms
- `phase4_only` — scale/crop transforms, with CutMix handled at batch level

Each phase is evaluated independently against the baseline before cumulative phase combinations are studied in the ablation notebook.

## Smoke-test strategy

A smoke-test mode is used before full experimentation to verify that:

- augmentation modules load correctly
- transforms apply cleanly to the processed dataset
- dataloaders, model forward passes, and training loops run without errors
- saved metrics, predictions, and figures are produced as expected

Smoke mode uses a stratified subset of the training and validation data, reduced epoch count, and a single backbone to confirm that the augmentation pipeline is functioning correctly before full runs are launched.

## Reproducibility

All runs should use a fixed seed, saved configuration, and explicit experiment labels so that smoke tests and full experiments can be repeated consistently.

In [ ]:
SMOKE_MODE = True
SMOKE_BACKBONES = ["resnet50"]
PHASE_EXPERIMENTS = ["phase2_only", "phase3_only", "phase4_only"]

SMOKE_TRAIN_FRACTION = 0.125
SMOKE_VAL_FRACTION = 0.25
SMOKE_MIN_TRAIN_PER_CLASS = 40
SMOKE_MIN_VAL_PER_CLASS = 15
SMOKE_MAX_TRAIN_PER_CLASS = 250
SMOKE_MAX_VAL_PER_CLASS = 75

EPOCHS = 1

In [ ]:
## When ready, switch to:

# SMOKE_MODE = False
# BACKBONES = ["resnet50", "efficientnet_b0", "densenet121"]
# EPOCHS = 5

In [ ]:
from pathlib import Path
import json
import random
import numpy as np
import torch

EXPERIMENT_SEED = 42
RUN_MODE = "smoke" if SMOKE_MODE else "full"

EXPERIMENT_CONFIG = {
    "notebook": "03_augmentation_experiments",
    "run_mode": RUN_MODE,
    "experiment_seed": EXPERIMENT_SEED,
    "smoke_mode": SMOKE_MODE,
    "smoke_backbones": SMOKE_BACKBONES if SMOKE_MODE else [],
    "phase_experiments": PHASE_EXPERIMENTS,
    "smoke_train_fraction": SMOKE_TRAIN_FRACTION,
    "smoke_val_fraction": SMOKE_VAL_FRACTION,
    "smoke_min_train_per_class": SMOKE_MIN_TRAIN_PER_CLASS,
    "smoke_min_val_per_class": SMOKE_MIN_VAL_PER_CLASS,
    "smoke_max_train_per_class": SMOKE_MAX_TRAIN_PER_CLASS,
    "smoke_max_val_per_class": SMOKE_MAX_VAL_PER_CLASS,
    "epochs": EPOCHS,
}

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name.lower() == "code" else cwd

OUT_ROOT = ROOT / "outputs"
CONFIG_DIR = OUT_ROOT / "configs" / "03_augmentation_experiments"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

config_path = CONFIG_DIR / f"{RUN_MODE}_config.json"
with open(config_path, "w") as f:
    json.dump(EXPERIMENT_CONFIG, f, indent=2)

print("Experiment seed:", EXPERIMENT_SEED)
print("Run mode:", RUN_MODE)
print("Saved config to:", config_path)

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


seed_everything(EXPERIMENT_SEED)

loader_generator = torch.Generator()
loader_generator.manual_seed(EXPERIMENT_SEED)

print("Global seed set.")
print("DataLoader generator seeded.")